# SpectraShift Week 7: frozen foundation probes
Use T4 x2 with Internet off and GPU 0. Attach source v6, Week 2 frozen data, Week 5 contracts, Week 6 contracts, Week 7 contracts, and Week 7 pilots.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week7.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 7 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week7-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 7 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

def install_offline_foundation_dependencies():
    wheels = sorted(INPUT.rglob('foundation-wheels')) + sorted(Path('/kaggle/working').rglob('foundation-wheels'))
    if not wheels: return
    missing = []
    for module, package in [('upath','universal-pathlib'), ('omegaconf','omegaconf'), ('iopath','iopath'), ('fvcore','fvcore'), ('einops','einops'), ('huggingface_hub','huggingface_hub')]:
        try: __import__(module)
        except ImportError: missing.append(package)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(wheels[0]), *missing])

install_offline_foundation_dependencies()
WORK = Path('/kaggle/working/spectrashift-week7-probes')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
WEEK7_CONTRACTS = unique_file('week7_contracts_summary.json')
RGB_CONTRACT = unique_file('rgb_percentile_contract.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
config = yaml.safe_load((PROJECT / 'configs/downstream/week7.yaml').read_text())
config['data'].update({'manifest_path': str(MANIFEST), 'staged_root': str(STAGED), 'normalization_path': str(NORMALIZATION), 'freeze_summary_path': str(FREEZE)})
config['contracts'].update({'week5_contracts_summary_path': str(WEEK5_CONTRACTS), 'rgb_contract_path': str(RGB_CONTRACT)})
RUNTIME_CONFIG = WORK / 'week7.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK)})

PILOTS = unique_file('week7_pilot_summary.json')


In [ ]:
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select GPU T4 x2'
from spectrashift.train.week7 import run_week7_probes

summary = run_week7_probes(RUNTIME_CONFIG, WORK, WEEK5_CONTRACTS, WEEK7_CONTRACTS, PILOTS)
print(json.dumps({key: value for key, value in summary.items() if key not in {'linear_runs','knn_runs','feature_caches'}}, indent=2))
assert summary['foundation_feature_cache_count'] == 2
assert summary['foundation_linear_probe_count'] == 36
assert summary['foundation_knn_run_count'] == 6
assert summary['evaluation_labels_loaded'] is False
